In [ ]:
import ray
from ray import serve
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import pyarrow as pa
import requests
from starlette.requests import Request
import json
from sentence_transformers import SentenceTransformer
import textdistance
import numpy as np

## Productionizing with Anyscale Services

To deploy into Anyscale's production-ready environment and featureset, we start by duplicating the Ray Serve config file.

Next, we add Anyscale entries:
* service name
* image URI
* compute config
* cloud
* working dir

See 
* https://docs.anyscale.com/services/deploy
* https://docs.anyscale.com/reference/service-api#serviceconfig

Notes:

1. the new "global" working dir must include the service script (import path)
1. service description, code, and dependencies should have no requirements/dependenies on the workspace (if you are developing in a workspace)

```yaml
name: search_recommend_c26_61

image_uri: anyscale/image/c26:5
compute_config: head-2a10g-small:2
working_dir : "https://anyscale-materials.s3.us-west-2.amazonaws.com/recommend_full_cached_model.zip"
cloud: education-us-west-2
query_auth_token_enabled: True

applications:
- name: search_recommend
  route_prefix: /
  import_path: search_and_recommend:bound_ingress
  deployments:
  - name: DatabaseFacade
    num_replicas: 3
    graceful_shutdown_wait_loop_s: 2.0
    graceful_shutdown_timeout_s: 20.0
    health_check_period_s: 10.0
    health_check_timeout_s: 30.0
        
  - name: Recommender
    num_replicas: 2
    max_ongoing_requests: 100
    ray_actor_options:
      num_cpus: 1
      num_gpus: 0.5
      
  - name: SemanticSearch
    autoscaling_config:
      min_replicas: 2
      max_replicas: 4

  - name: Ingress
    autoscaling_config:
      min_replicas: 2
      max_replicas: 4 
```

In [ ]:
! pwd

In [ ]:
! anyscale service deploy --config-file anyscale_serve_config.yaml

We can check the startup progress in the Anyscale UI and dashboards.

Once the service is running, the Query button in the UI or the `anyscale service status` command gives us params for querying the service

In [ ]:
! anyscale service status --name search_recommend_c26_61

In [ ]:
# Service specific config
base_url = "<YOUR_SERVICE_URL>"
token = "<YOUR_QUERY_AUTH_TOKEN>"

# Requests config
path = "/"
full_url = f"{base_url}{path}"
headers = {"Authorization": f"Bearer {token}"}

In [ ]:
resp = requests.post(full_url, headers=headers, json='{ "id" : "7034dd99-ceb3-474d-a0ba-5beaf122273f", "query" : "steel bowls"}')

resp.json()

We can modify our config or code and deploy again to trigger Anyscale's "Canary Rollout" feature, which will deploy the new version of the service alongside the old one, transition traffic to it gradually, and allow us to roll back.

To manually route a request to a specific version for testing, we can add a header with the specific version ID or "canary" / "primary"

> Note: version routing only operates during the rollout

In [ ]:
# Service specific config
base_url = "<YOUR_SERVICE_URL>"
token = "<YOUR_QUERY_AUTH_TOKEN>"

# Requests config
path = "/"
full_url = f"{base_url}{path}"
headers = {
    "Authorization": f"Bearer {token}",
    "X-ANYSCALE-VERSION": "canary"
}

In [ ]:
resp = requests.post(full_url, headers=headers, json='{ "id" : "7034dd99-ceb3-474d-a0ba-5beaf122273f", "query" : "snacks"}')

resp.json()

In [ ]:
! anyscale service terminate --name search_recommend_c26_61